In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import json
from datetime import *

In [0]:
# Read the Parquet files in the New folder
df = spark.read.parquet(f"/Volumes/currency/landing/currency_analysis/Quotes/New/*.parquet")

df.createOrReplaceTempView("df")

df = spark.sql("""
    SELECT
        CAST(dataHoraCotacao AS DATE) AS dt_period,
        currency AS dc_currency,
        cotacaoCompra AS vl_quote
    FROM
        df
    ORDER BY dt_period ASC
""").dropDuplicates(["dc_currency", "dt_period"])

display(df)

In [0]:
%sql
-- Create the table if it does not exist yet
CREATE TABLE IF NOT EXISTS currency.`02_silver_currency`.quotes (
    dt_period DATE,
    dc_currency STRING,
    vl_quote DOUBLE
)
USING DELTA
PARTITIONED BY (dc_currency)

In [0]:
# Merge the new rows into the quotes table
df.createOrReplaceTempView("df_new")

spark.sql("""
    MERGE INTO currency.`02_silver_currency`.quotes AS quotes
    USING (
        SELECT
            dt_period,
            dc_currency,
            vl_quote
        FROM
            df_new
    ) AS new_rows
    ON quotes.dc_currency = new_rows.dc_currency
        AND quotes.dt_period = new_rows.dt_period
    WHEN NOT MATCHED THEN
        INSERT (dt_period, dc_currency, vl_quote)
        VALUES (new_rows.dt_period, new_rows.dc_currency, new_rows.vl_quote)
""")

In [0]:
source = "/Volumes/currency/landing/currency_analysis/Quotes/New/"
destination = "/Volumes/currency/landing/currency_analysis/Quotes/Processed/"

# create the destination folder if it does not exist yet
try:
    dbutils.fs.ls(destination)
except Exception:
    dbutils.fs.mkdirs(destination)

# move all the content recursively
dbutils.fs.mv(source, destination, recurse=True)